# 06 - 模块与包（Java vs Python）

Python 的 import 机制和 Java 差别不小，项目结构也完全不同。

## 今日目标（15-20 分钟）

- 掌握 import 的各种写法
- 理解 `__init__.py` 的作用
- 了解 Python 项目结构约定
- 区分相对导入和绝对导入

完成标准：通过末尾打卡题


## 1. import 基础


In [ ]:
# 说明：演示 import 整个模块与 as 别名的两种基础导入方式。

# Java: import java.util.List;
# Python: import 模块名

# 导入整个模块
import json
print(json.dumps({"a": 1}))  # 用 模块名.函数名 调用

# 导入模块并起别名（Java 没有）
import json as j
print(j.dumps({"b": 2}))


In [ ]:
# 说明：演示 from ... import ... 的选择性导入与可读性权衡。

# from ... import ...：只导入需要的东西
# Java: import java.util.List; （Java 总是这种风格）

from pathlib import Path
from json import dumps, loads

print(Path(".").resolve())
print(dumps({"c": 3}))  # 不用写 json.dumps，直接 dumps


In [ ]:
# 说明：本段示例对应「1. import 基础」，建议先看注释再运行，重点观察输出与预期是否一致。

# from ... import *：导入模块所有公开内容（不推荐，污染命名空间）
# 相当于 Java 的 import java.util.*;

from math import *
print(sqrt(16))  # 能直接用，但不知道 sqrt 从哪来
print(pi)

# 实际开发中避免 import *，显式导入可读性更好

# ⚠️ 坑：import * 的最大问题是静默覆盖——
# from moduleA import *
# from moduleB import *   # 如果 moduleB 也有同名函数，前面的被无声替换，不报错不提示
#
# 例如：from os.path import *  再 from posixpath import *
# 很难排查哪个 join() 最终生效
# IDE 也无法追踪函数来源，自动补全和跳转定义都会失效


## 2. 模块就是文件

Python 里**一个 .py 文件就是一个模块**，文件名就是模块名。

```
# Java 的类和文件名必须一致：User.java → public class User
# Python 没有这个限制：utils.py 里可以有任意多个类和函数
```

```python
# utils.py
def add(a, b):
    return a + b

PI = 3.14

# main.py
from utils import add, PI
print(add(1, 2))
```


## 3. 包（Package）和 `__init__.py`

**包通常是一个目录（推荐包含 `__init__.py`）**，对比 Java 的 package。

```
Java 项目结构：
src/main/java/
  └── com/example/
        ├── service/
        │   └── UserService.java
        └── model/
            └── User.java

Python 项目结构：
my_project/
  ├── __init__.py          ← 让 my_project 成为一个包
  ├── service/
  │   ├── __init__.py      ← 让 service 成为子包
  │   └── user_service.py
  └── model/
      ├── __init__.py
      └── user.py
```


In [ ]:
# 说明：本段示例对应「3. 包（Package）和 __init__.py」，建议先看注释再运行，重点观察输出与预期是否一致。

# __init__.py 的作用：

# 1. 标记目录为包（Python 3.3+ 可省略，但建议保留）
# 2. 控制 from package import * 时导出哪些内容
# 3. 包初始化代码（导入时自动执行）

# __init__.py 示例：
# my_project/service/__init__.py
#
# from .user_service import UserService   ← 重新导出，简化外部 import 路径
#
# 这样外部可以写：
# from my_project.service import UserService
# 而不用写：
# from my_project.service.user_service import UserService

print("__init__.py 常见用法：重新导出 + 简化导入路径")

# 💡 实战：FastAPI 项目里 routers/__init__.py 常见写法——
# routers/__init__.py:
#   from .expression import router as expression_router
#   from .user import router as user_router
#
# main.py:
#   from app.routers import expression_router, user_router
#   app.include_router(expression_router)
#   app.include_router(user_router)
#
# main.py 只需一行 import，不用知道 router 在哪个具体文件里
# 新增路由时只改 routers/__init__.py，main.py 不动


## 4. 相对导入 vs 绝对导入


In [ ]:
# 说明：演示绝对导入与相对导入的使用场景与常见限制。

# 绝对导入：从项目根目录开始（推荐，清晰明确）
# from my_project.service.user_service import UserService
# from my_project.model.user import User

# 相对导入：用 . 表示当前包，.. 表示上级包（只能在包内部使用）
# 假设在 my_project/service/user_service.py 里：
# from .helper import format_name       ← 同级目录
# from ..model.user import User          ← 上级目录的 model 包

# 注意：相对导入不能在直接运行的脚本中使用（python xxx.py），
# 只能在被 import 的模块中使用

# Java 没有相对导入，总是用全限定包名

print("绝对导入更清晰，相对导入在包内部用")

# ⚠️ 坑：用 python xxx.py 直接运行含相对导入的文件会报错——
# ImportError: attempted relative import with no known parent package
# 原因：直接运行时 Python 不知道这个文件属于哪个包（__name__ == "__main__"，没有包上下文）
#
# 解决方案：从项目根目录用 -m 标志以模块方式运行：
# cd /home/kai/projects/say-right
# python -m app.service.user_service   ← 用点分路径，不用文件路径
#
# 这也是为什么 FastAPI 启动命令是 uvicorn app.main:app，而不是 python app/main.py


## 5. `if __name__ == "__main__"`


In [ ]:
# 说明：演示脚本入口约定，区分“直接运行”与“被导入”两种执行场景。

# Java: public static void main(String[] args) { ... }
# Python: 没有 main 方法，整个文件从上到下执行

# 但有个约定：

# utils.py
# def add(a, b):
#     return a + b
#
# if __name__ == "__main__":
#     # 只有直接运行 python utils.py 时才执行
#     # 被别人 import 时不会执行
#     print(add(1, 2))

# __name__ 的值：
# - 直接运行时："__main__"
# - 被 import 时：模块名（如 "utils"）

print(f"当前 __name__ = {__name__}")  # 在 notebook 里是 __main__


## 6. 实际 FastAPI 项目结构示例

```
say-right/
├── .venv/                    ← 虚拟环境（不提交到 git）
├── requirements.txt          ← 依赖清单
├── app/
│   ├── __init__.py
│   ├── main.py               ← FastAPI 入口：app = FastAPI()
│   ├── config.py             ← 配置
│   ├── routers/              ← 路由（对比 Java 的 Controller）
│   │   ├── __init__.py
│   │   └── expression.py
│   ├── services/             ← 业务逻辑（对比 Java 的 Service）
│   │   ├── __init__.py
│   │   └── ai_service.py
│   ├── models/               ← 数据模型（对比 Java 的 Entity/DTO）
│   │   ├── __init__.py
│   │   └── expression.py
│   └── utils/                ← 工具函数
│       ├── __init__.py
│       └── text.py
└── tests/
    ├── __init__.py
    └── test_expression.py
```

对比 Java 的 Spring Boot 分层：

| Java Spring | Python FastAPI |
|-------------|----------------|
| `@RestController` | `routers/` |
| `@Service` | `services/` |
| `@Entity` / DTO | `models/`（pydantic） |
| `application.yml` | `config.py` / `.env` |
| `src/test/java/` | `tests/` |


## 7. Java 对照速记

| Java | Python |
|------|--------|
| `import java.util.List;` | `from typing import List`（类型注解场景） |
| `import java.util.*;` | `from module import *`（不推荐） |
| 无 | `import json as j`（别名） |
| 包 = 目录 + package 声明 | 包 = 目录（通常含 `__init__.py`） |
| 类名和文件名必须一致 | 无此限制，一个文件可以有多个类 |
| `public static void main` | `if __name__ == "__main__"` |
| 无 | 相对导入 `from . import xxx` |


## 今日打卡题

1. `import json` 和 `from json import dumps` 有什么区别？各适合什么场景？
2. `__init__.py` 的三个作用是什么？
3. 下面的项目结构中，`user_service.py` 要导入同级的 `helper.py` 和上级 `model/user.py`，分别怎么写？

```
app/
├── service/
│   ├── __init__.py
│   ├── user_service.py
│   └── helper.py
└── model/
    ├── __init__.py
    └── user.py
```


In [ ]:
# 说明：这是练习留空区域，建议先独立完成，再运行后续参考答案进行对照。

# TODO: 先自己写


In [ ]:
# 说明：这是打卡题参考实现，建议先自己做一遍，再用它核对思路和语法。

# 参考答案

# 1.
# import json          → 调用时写 json.dumps()，适合频繁使用多个函数
# from json import dumps → 直接写 dumps()，适合只用一两个函数

# 2.
# a. 标记目录为 Python 包
# b. 控制 from package import * 的导出内容
# c. 重新导出子模块内容，简化外部导入路径

# 3.
# 在 user_service.py 中：
# from .helper import some_function          ← 同级（相对导入）
# from ..model.user import User              ← 上级的 model 包（相对导入）
# 或绝对导入：
# from app.service.helper import some_function
# from app.model.user import User

print("06 done!")


下一步建议：进入 `07-pythonic-features.ipynb`，掌握列表推导式、生成器等 Python 独有写法。
